# WTI Crude Oil — The Forecast Tool (Notebook 5)

Notebook 2 built a **progressive capability staircase** for the analyst agent:
no tools → news → news + open-ended code execution. This notebook adds a
**fourth, contrasting capability level**: a conventional **function tool**.

Instead of letting the agent write arbitrary Python, we expose a single,
rigidly-typed callable — `run_forecast` — that fits a pre-specified statistical
model (**AutoARIMA**) up to a cutoff date and returns a structured forecast. The
agent expresses intent through the tool's parameters (`series_id`,
`cutoff_date`, `horizons`, `frequency`); the series data never enters the LLM
context window.

| Path | Mechanism | Trade-off |
|------|-----------|-----------|
| `build_wti_code_exec_config` | Open-ended code generation | Maximum flexibility, less control |
| `build_wti_tool_config` (this NB) | Fixed function tool | Less flexibility, full control + reproducibility |

> **Prerequisite:** Read [`02_intro_agentic_predictor.ipynb`](02_intro_agentic_predictor.ipynb)
> first for the staircase framing and the `AgentPredictor` interface.

In [1]:
# ── Environment setup: load .env, corporate CA, build-cli token refresh ──────
import os
import shutil
import subprocess
import sys
from pathlib import Path

from dotenv import load_dotenv


def _find_repo_root(start=None):
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "pyproject.toml").exists() and (cand / "aieng-forecasting").is_dir():
            return cand
    return Path.cwd().resolve().parents[1]


ROOT = _find_repo_root()
load_dotenv(ROOT / ".env")

# curl_cffi (used by yfinance) bundles its own libcurl and ignores
# pip-system-certs. On macOS, point it at a bundle that includes the corporate CA.
if sys.platform == "darwin" and not os.environ.get("CURL_CA_BUNDLE"):
    import certifi as _certifi

    _combined = Path.home() / ".cache" / "agentic-forecasting" / "combined-ca.pem"
    _combined.parent.mkdir(parents=True, exist_ok=True)
    _kc = subprocess.run(
        ["security", "find-certificate", "-a", "-p", "/Library/Keychains/System.keychain"],
        capture_output=True, text=True,
    )
    _combined.write_text(Path(_certifi.where()).read_text() + "\n" + _kc.stdout)
    os.environ["CURL_CA_BUNDLE"] = str(_combined)
    os.environ["SSL_CERT_FILE"] = str(_combined)
    os.environ["REQUESTS_CA_BUNDLE"] = str(_combined)

# build-cli tokens are short-lived; refresh so GUI-launched kernels (where
# ~/.local/bin is not on PATH) don't silently use a stale key.
_BUILDCLI = next(
    (str(p) for p in [Path.home() / ".local" / "bin" / "build-cli", Path(shutil.which("build-cli") or "")] if p.exists()),
    None,
)
if "build-cli.roche.com" in os.environ.get("PROXY_BASE_URL", ""):
    if _BUILDCLI:
        _result = subprocess.run([_BUILDCLI, "auth", "token"], capture_output=True, text=True, timeout=15)
        if _result.returncode == 0 and _result.stdout.strip():
            os.environ["PROXY_API_KEY"] = _result.stdout.strip()
            print(f"build-cli token refreshed: …{_result.stdout.strip()[-4:]}")
        else:
            print(f"⚠️  build-cli auth token failed: {_result.stderr.strip() or 'no output'}")
            print("    Run 'build-cli login' in a terminal, then restart the kernel.")
    else:
        print("⚠️  build-cli not found — set PROXY_API_KEY manually: export PROXY_API_KEY=$(build-cli auth token)")


build-cli token refreshed: …tQsA


---
## 1. Setup & Data Registration

In [2]:
import warnings


warnings.filterwarnings("ignore")

import pandas as pd
from aieng.forecasting.evaluation.task import ForecastingTask
from energy_oil_forecasting.data import WTI_SERIES_ID, build_wti_service


# The data service is shared by the tool and the prompt builder. The tool reads
# series data directly from it (server-side) — it is never sent to the model.
data_service = build_wti_service()

AS_OF = pd.Timestamp("2026-03-01")  # information cutoff (context available)
ORIGIN = pd.Timestamp("2026-03-02")  # the day we forecast *from*

ctx = data_service.context(as_of=AS_OF)
full_df = ctx.get_series(WTI_SERIES_ID)

print(f"Trading days in cache up to {AS_OF.date()}: {len(full_df)}")
print(f"Last WTI close: ${full_df['value'].iloc[-1]:.2f}/bbl on {str(full_df['timestamp'].iloc[-1])[:10]}")

Trading days in cache up to 2026-03-01: 5569
Last WTI close: $65.21/bbl on 2026-02-26


In [3]:
task = ForecastingTask(
    task_id="wti_oil_price_forecast",
    target_series_id=WTI_SERIES_ID,
    horizons=[5, 10, 21],
    frequency="B",
    description="WTI Crude Oil front-month futures — 5, 10, 21 business days ahead.",
)

print("Task:", task.task_id, "| horizons:", task.horizons, "| as_of:", ctx.as_of)

Task: wti_oil_price_forecast | horizons: [5, 10, 21] | as_of: 2026-03-01 00:00:00


---
## 2. The tool, standalone

`ForecastTool` is deterministic and needs no LLM. We call it directly here to
show exactly what the agent will receive: a JSON block with point forecasts and
prediction intervals per horizon, plus the series metadata and the cutoff date
used.

We pass an explicit `data_service` (the one registered above) so the tool reads
from the same cache. The tool wraps a `Predictor`; here we inject an AutoARIMA
predictor with a modest `num_samples` (it is slow per origin).

In [4]:
from aieng.forecasting.methods.agentic import ForecastTool
from aieng.forecasting.methods.numerical.darts_arima import DartsAutoARIMAPredictor


tool = ForecastTool(data_service, predictor=DartsAutoARIMAPredictor(num_samples=200))

print("Running AutoARIMA forecast (this can take tens of seconds)...")
result_json = tool.run_forecast(
    series_id=WTI_SERIES_ID,
    cutoff_date=str(AS_OF.date()),
    horizons=task.horizons,
    frequency="B",
)
print(result_json)

Running AutoARIMA forecast (this can take tens of seconds)...


{
  "status": "ok",
  "series_id": "wti_crude_oil_price",
  "series_description": "WTI Crude Oil continuous front-month futures adjusted close (Yahoo Finance CL=F)",
  "units": "USD/bbl",
  "frequency": "B",
  "cutoff_date": "2026-03-01",
  "n_observations_at_cutoff": 5569,
  "last_observed": {
    "date": "2026-02-26",
    "value": 65.20999908447266
  },
  "forecasts": [
    {
      "horizon": 5,
      "forecast_date": "2026-03-06",
      "point_forecast": 65.35278378700153,
      "intervals": {
        "80%": {
          "lower": 59.90963998601698,
          "upper": 70.23397881734435
        },
        "90%": {
          "lower": 58.561754386618055,
          "upper": 71.3464632661117
        }
      },
      "quantiles": {
        "0.05": 58.561754386618055,
        "0.1": 59.90963998601698,
        "0.2": 61.713231526670896,
        "0.3": 63.038996460858094,
        "0.4": 64.13667624130684,
        "0.5": 65.35278378700153,
        "0.6": 66.27902004651827,
        "0.7": 67.137

Note the `notes` field: a true 95% interval is not reported because the
predictor's standard quantile grid tops out at p05/p95, so the widest honest
interval is **90%** (p05–p95). The tool reports the **80%** (p10–p90) and
**90%** intervals plus the full quantile grid — it never fabricates coverage
the model did not produce.

---
## 3. Wiring the tool into the agent

`build_wti_tool_config()` is the fourth capability factory. It combines the
bounded Google Search sub-agent (temporal cutoff enforced) with the forecast
tool, and appends an instruction supplement telling the agent to call
`run_forecast` once before producing its forecast.

We pass the same `data_service` so the config does not rebuild it.

In [5]:
from energy_oil_forecasting.analyst_agent import (
    build_wti_agent_predictor,
    build_wti_tool_config,
)


# Models: "anthropic/claude-haiku-4-5-20251001" (lite) · "anthropic/claude-sonnet-4-6[1m]" (advanced)
tool_config = build_wti_tool_config(
    model="anthropic/claude-sonnet-4-6[1m]",
    # model="anthropic/claude-haiku-4-5-20251001",  # lite
    data_service=data_service,
    num_samples=200,
)

print("=== Tool config summary ===")
print("name:                ", tool_config.name)
print("model:               ", tool_config.model)
print("function_tools:      ", len(tool_config.function_tools))
print("context_retrieval:   ", tool_config.context_retrieval.enabled)
print("cutoff enforced:     ", tool_config.context_retrieval.enforce_cutoff)
print("\n=== Forecast tool supplement (tail of instruction) ===")
print(tool_config.instruction[-700:])

=== Tool config summary ===
name:                 wti_analyst_tool
model:                anthropic/claude-sonnet-4-6[1m]
function_tools:       1
context_retrieval:    True
cutoff enforced:      True

=== Forecast tool supplement (tail of instruction) ===
orecast you can reason from.

Call it ONCE before producing your forecast, with:
- `series_id`: "wti_crude_oil_price"
- `cutoff_date`: the `as_of` date from the payload (YYYY-MM-DD). This is the
  information cutoff — the model uses only data on or before it.
- `horizons`: the `horizons` list from the payload.
- `frequency`: "B" (WTI trades on business days).

The tool returns JSON with point forecasts and 80%/90% prediction intervals per
horizon. Treat it as a disciplined statistical anchor: combine it with the
market context from the search sub-agent. You may adjust away from the baseline
when fundamentals or geopolitical risk justify it — document your reasoning in
the `rationale` fields.


---
## 4. A single agent call

Wrapping the config in an `AgentPredictor` and calling `predict` runs **one**
agent turn. In that turn the agent calls the Google Search sub-agent for market
context **and** `run_forecast` for the AutoARIMA anchor, then returns a
structured forecast that conditions on both.

In [6]:
tool_predictor = build_wti_agent_predictor(tool_config)

print(f"Predictor ID: {tool_predictor.predictor_id}")
print("Running tool-equipped agent... (Google Search + AutoARIMA forecast tool)")

tool_preds = tool_predictor.predict(task, ctx)

print("\nTool-equipped agent forecast:\n")
for p in tool_preds:
    fc = p.payload
    print(
        f"  h={task.horizons[tool_preds.index(p)]:>2}d  "
        f"point=${fc.point_forecast:.2f}  "
        f"80%CI=[${fc.quantiles[0.10]:.2f}, ${fc.quantiles[0.90]:.2f}]"
    )
if tool_preds and tool_preds[0].metadata.get("rationale"):
    print("\nAgent rationale:", tool_preds[0].metadata["rationale"][:500])

Predictor ID: agent_predictor_wti_analyst_tool_continuous
Running tool-equipped agent... (Google Search + AutoARIMA forecast tool)

Provider List: https://docs.litellm.ai/docs/providers




Provider List: https://docs.litellm.ai/docs/providers




Provider List: https://docs.litellm.ai/docs/providers




Tool-equipped agent forecast:

  h= 5d  point=$74.50  80%CI=[$65.50, $85.00]
  h=10d  point=$80.00  80%CI=[$65.00, $96.00]
  h=21d  point=$82.00  80%CI=[$64.50, $104.00]

Agent rationale: This forecast reflects an extreme geopolitical shock that erupted on the as_of date (March 1, 2026), completely superseding the statistical AutoARIMA baseline. Key facts driving the upward adjustment: (1) US/Israel launched coordinated strikes on Iran on Feb 28, 2026; (2) Iran declared the Strait of Hormuz CLOSED on March 1 — a chokepoint handling ~20% of global oil trade (~20M b/d); (3) WTI already surged 8.4% to $72.74/bbl on March 2, with Brent reaching $87+ by March 7; (4) Iran struck Gulf n


---
## 5. Wrap-up

- The tool-equipped agent returns standard `Prediction` objects, so it drops
  straight into the Notebook 4 backtest harness via
  `build_wti_agent_predictor(build_wti_tool_config(...))`.
- **Conventional tools vs. code generation** is a deliberate design divergence:
  the tool path trades flexibility for a fixed, auditable, reproducible
  interface — arguably a safer way to grant an agent a new ability.
- AutoARIMA is just one example. `ForecastTool` wraps any `Predictor` (passed at
  construction), so swapping in Prophet, ETS, or an ensemble needs no signature
  change. And `series_id` makes the tool reusable for food CPI, the BoC rate,
  and other registered series.